# Sprint 13 Project — Customer Churn Prediction


* Complete analysis of gym customer churn, including EDA, predictive models, and clustering.


## Clarifying Questions

Before starting the analysis, a few important questions were considered:

1. Which customer characteristics are most associated with churn?
2. Is it possible to predict which customers are more likely to cancel?
3. Are there distinct groups of customers with similar behaviors?
4. Which segments show the highest churn risk?
5. What retention actions can be recommended based on the results?

## Work Plan

1. Load and inspect the data.
2. Perform exploratory data analysis (EDA) to identify patterns related to churn.
3. Build predictive models to forecast customer churn.
4. Segment customers using clustering techniques.
5. Develop strategic recommendations to reduce churn.

## Step 1. Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

# Load the dataset
df = pd.read_csv('gym_churn_us.csv')
print(f"Shape: {df.shape}")
df.head()

## Step 2. Exploratory Data Analysis (EDA)

### 2.1 Missing Values Check and Descriptive Statistics

In [ ]:
# Missing values
print("=== Missing Values ===")
print(df.isnull().sum())
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

In [ ]:
# Check for duplicate rows
print(f"Duplicate rows: {df.duplicated().sum()}")

In [ ]:
# Descriptive statistics
print("=== Descriptive Statistics ===")
df.describe().round(2)

### 2.2 Group Means (Churn vs. Non-Churn)

In [ ]:
group_means = df.groupby('Churn').mean().round(3).T
group_means.columns = ['Stayed (0)', 'Churned (1)']
group_means['Difference (%)'] = ((group_means['Churned (1)'] - group_means['Stayed (0)']) / group_means['Stayed (0)'] * 100).round(1)
print(group_means.to_string())

### 2.3 Feature Distribution by Churn Group

In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(18, 22))
axes = axes.flatten()

features = [c for c in df.columns if c != 'Churn']
colors = ['#2ecc71', '#e74c3c']

for i, feat in enumerate(features):
    ax = axes[i]
    if df[feat].nunique() <= 4:
        # Categorical / discrete variable — bar chart
        stayed = df[df['Churn']==0][feat].value_counts(normalize=True).sort_index()
        churned = df[df['Churn']==1][feat].value_counts(normalize=True).sort_index()
        x = np.arange(len(stayed))
        ax.bar(x - 0.2, stayed.values, 0.4, label='Stayed', color=colors[0], alpha=0.85)
        ax.bar(x + 0.2, churned.values, 0.4, label='Churned', color=colors[1], alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(stayed.index)
        ax.set_ylabel('Proportion')
    else:
        # Continuous variable — KDE
        df[df['Churn']==0][feat].plot.kde(ax=ax, color=colors[0], label='Stayed', linewidth=2)
        df[df['Churn']==1][feat].plot.kde(ax=ax, color=colors[1], label='Churned', linewidth=2)
        ax.set_ylabel('Density')
    ax.set_title(feat, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)

# Remove extra axes
for j in range(len(features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distribution by Churn Group', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('churn_distributions.png', bbox_inches='tight')
plt.show()

### 2.4 Correlation Matrix

In [ ]:
plt.figure(figsize=(14, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5, annot_kws={"size": 8})
plt.title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_matrix.png', bbox_inches='tight')
plt.show()

# Correlations with Churn
print("\n=== Correlation with Churn (sorted) ===")
print(corr['Churn'].drop('Churn').sort_values(key=abs, ascending=False).round(3))

## Step 3. Churn Predictive Models

In [ ]:
# Separate features and target
X = df.drop('Churn', axis=1)
y = df['Churn']

# Train/validation split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")
print(f"Churn rate in train: {y_train.mean():.2%}")
print(f"Churn rate in test:  {y_test.mean():.2%}")

In [ ]:
# Logistic Regression
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

acc_lr  = accuracy_score(y_test, y_pred_lr)
prec_lr = precision_score(y_test, y_pred_lr)
rec_lr  = recall_score(y_test, y_pred_lr)

print("=== Logistic Regression ===")
print(f"Accuracy:  {acc_lr:.4f}")
print(f"Precision: {prec_lr:.4f}")
print(f"Recall:    {rec_lr:.4f}")
print()
print(classification_report(y_test, y_pred_lr, target_names=['Stayed','Churned']))

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

acc_rf  = accuracy_score(y_test, y_pred_rf)
prec_rf = precision_score(y_test, y_pred_rf)
rec_rf  = recall_score(y_test, y_pred_rf)

print("=== Random Forest ===")
print(f"Accuracy:  {acc_rf:.4f}")
print(f"Precision: {prec_rf:.4f}")
print(f"Recall:    {rec_rf:.4f}")
print()
print(classification_report(y_test, y_pred_rf, target_names=['Stayed','Churned']))

In [ ]:
# Baseline — DummyClassifier (majority class)
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

acc_dummy  = accuracy_score(y_test, y_pred_dummy)
prec_dummy = precision_score(y_test, y_pred_dummy, zero_division=0)
rec_dummy  = recall_score(y_test, y_pred_dummy, zero_division=0)

print("=== Baseline — DummyClassifier (always predicts the majority class) ===")
print(f"Accuracy:  {acc_dummy:.4f}")
print(f"Precision: {prec_dummy:.4f}")
print(f"Recall:    {rec_dummy:.4f}")
print()
print(classification_report(y_test, y_pred_dummy, target_names=['Stayed','Churned'], zero_division=0))
print("Note: a model that always predicts 'does not churn' is already right ~73.5% of the time,")
print("since that is the proportion of the majority class. The trained models need to")
print("beat this reference to demonstrate real learning.")

In [ ]:
# Visual comparison — models vs baseline
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics = ['Accuracy', 'Precision', 'Recall']
lr_scores    = [acc_lr,    prec_lr,    rec_lr]
rf_scores    = [acc_rf,    prec_rf,    rec_rf]
dummy_scores = [acc_dummy, prec_dummy, rec_dummy]

x = np.arange(len(metrics))
width = 0.25

axes[0].bar(x - width, lr_scores,    width, label='Logistic Regression', color='#3498db', alpha=0.85)
axes[0].bar(x,         rf_scores,    width, label='Random Forest',       color='#e67e22', alpha=0.85)
axes[0].bar(x + width, dummy_scores, width, label='Baseline (Dummy)',    color='#95a5a6', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].set_ylim(0, 1.1)
axes[0].set_title('Model Metrics Comparison', fontweight='bold')
axes[0].legend()
for i, (lr_v, rf_v, dm_v) in enumerate(zip(lr_scores, rf_scores, dummy_scores)):
    axes[0].text(i - width, lr_v + 0.02, f'{lr_v:.3f}', ha='center', fontsize=8)
    axes[0].text(i,         rf_v + 0.02, f'{rf_v:.3f}', ha='center', fontsize=8)
    axes[0].text(i + width, dm_v + 0.02, f'{dm_v:.3f}', ha='center', fontsize=8)

# Feature importance (RF)
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)
feat_imp.plot.barh(ax=axes[1], color='#9b59b6', alpha=0.85)
axes[1].set_title('Feature Importance (Random Forest)', fontweight='bold')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight')
plt.show()

print("\n=== Conclusion ===")
print(f"Baseline (Dummy):     Accuracy = {acc_dummy:.4f}")
print(f"Logistic Regression:  Accuracy = {acc_lr:.4f}  (+{acc_lr - acc_dummy:.4f} vs baseline)")
print(f"Random Forest:        Accuracy = {acc_rf:.4f}  (+{acc_rf - acc_dummy:.4f} vs baseline)")
if acc_rf >= acc_lr:
    print("\nRandom Forest achieved better (or equivalent) performance than Logistic Regression.")
else:
    print("\nLogistic Regression achieved better performance than Random Forest.")
print("Both models clearly outperform the baseline in precision and recall for the 'Churned' class.")

# Feature Importance

In [ ]:
# Note: the rf model was already trained in Step 3 above.
# We use rf.feature_importances_ directly — no need to retrain.
# (redundant code removed)

In [ ]:

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)


plt.figure(figsize=(10,6))

sns.barplot(
    data=feature_importance.head(10),
    x='Importance',
    y='Feature',
    palette='Purples_r'
)

plt.title('Top 10 Factors Influencing Customer Churn')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
print("Main factors associated with churn:")
print(feature_importance.head(5))

## Step 4. Customer Clustering

### 4.1 Data Standardization

In [ ]:
# Exclude Churn column for clustering
X_clust = df.drop('Churn', axis=1)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clust)
X_scaled_df = pd.DataFrame(X_scaled, columns=X_clust.columns)
print("Standardized data. Shape:", X_scaled_df.shape)
X_scaled_df.describe().round(2)

### 4.2 Dendrogram (Hierarchical Clustering)

In [ ]:
# Sample for the dendrogram (avoid slowness with large n)
np.random.seed(42)
sample_idx = np.random.choice(len(X_scaled), size=200, replace=False)
X_sample = X_scaled[sample_idx]

linked = linkage(X_sample, method='ward')

# Identify the largest distance jump (hierarchical elbow)
last_merges = linked[-10:, 2]  # last 10 merges
acceleration = np.diff(last_merges, 2)  # second derivative
k_suggested = acceleration[::-1].argmax() + 2  # +2 since diff reduces the index
print(f"Number of clusters suggested by the largest distance jump: {k_suggested}")

plt.figure(figsize=(16, 6))
dendrogram(linked, truncate_mode='lastp', p=30, leaf_rotation=90,
           leaf_font_size=9, show_contracted=True)
# Cut line based on the largest observed jump
cut_height = (linked[-k_suggested, 2] + linked[-k_suggested+1, 2]) / 2
plt.axhline(y=cut_height, color='red', linestyle='--', alpha=0.7,
            label=f'Suggested cut ({k_suggested} clusters)')
plt.title('Dendrogram (Ward, sample of 200 customers)', fontsize=13, fontweight='bold')
plt.xlabel('Cluster Index')
plt.ylabel('Distance')
plt.legend()
plt.tight_layout()
plt.savefig('dendrogram.png', bbox_inches='tight')
plt.show()
print(f"\nThe dendrogram suggests {k_suggested} distinct customer groups.")

In [ ]:
from sklearn.metrics import silhouette_score

# Elbow Method + Silhouette to choose k
inertia = []
sil_scores = []
k_range = range(2, 9)

for k in k_range:
    km_tmp = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km_tmp.fit_predict(X_scaled)
    inertia.append(km_tmp.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(list(k_range), inertia, 'o-', color='#3498db')
axes[0].set_title('Elbow Method (Inertia)', fontweight='bold')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')

axes[1].plot(list(k_range), sil_scores, 'o-', color='#e74c3c')
axes[1].set_title('Silhouette Score by k', fontweight='bold')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.savefig('elbow_silhouette.png', bbox_inches='tight')
plt.show()

best_k = list(k_range)[sil_scores.index(max(sil_scores))]
print(f"Best k by Silhouette Score: {best_k}")
print(f"Maximum silhouette: {max(sil_scores):.4f}")
print(f"(k chosen for K-Means: {best_k})")

### 4.3 K-Means with Optimized k

In [ ]:
km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['cluster'] = km.fit_predict(X_scaled)

print(f"K-Means trained with k={best_k} clusters (choice based on Silhouette Score)")
print("\nCluster distribution:")
print(df['cluster'].value_counts().sort_index())

### 4.4 Mean Values by Cluster

In [ ]:
cluster_means = df.groupby('cluster').mean().round(3)
print("=== Means by Cluster ===")
print(cluster_means.to_string())

In [ ]:
# Heatmap of cluster profiles
fig, ax = plt.subplots(figsize=(14, 6))
cluster_norm = cluster_means.drop('Churn', axis=1)
cluster_norm_scaled = (cluster_norm - cluster_norm.min()) / (cluster_norm.max() - cluster_norm.min())
sns.heatmap(cluster_norm_scaled.T, annot=cluster_norm.T.round(2), fmt='g',
            cmap='YlOrRd', linewidths=0.5, ax=ax, annot_kws={"size": 8})
ax.set_title('Normalized Cluster Profile (real values annotated)', fontsize=13, fontweight='bold')
ax.set_xlabel('Cluster')
plt.tight_layout()
plt.savefig('cluster_heatmap.png', bbox_inches='tight')
plt.show()

### 4.5 Feature Distribution by Cluster

In [ ]:
# 4.5 Feature distribution by cluster

features_plot = [
    'Age',
    'Lifetime',
    'Contract_period',
    'Month_to_end_contract',
    'Avg_class_frequency_total',
    'Avg_class_frequency_current_month',
    'Avg_additional_charges_total'
]

fig, axes = plt.subplots(3, 3, figsize=(16, 14))
axes = axes.flatten()

# Palette based on the actual number of clusters
palette = sns.color_palette('tab10', best_k)

for i, feat in enumerate(features_plot):

    ax = axes[i]

    for c in sorted(df['cluster'].unique()):

        cluster_data = df[df['cluster'] == c][feat]

        # Avoid errors when a cluster has very few records
        if len(cluster_data) > 1:
            cluster_data.plot.kde(
                ax=ax,
                label=f'Cluster {c}',
                color=palette[c],
                linewidth=2
            )

    ax.set_title(feat, fontweight='bold')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

# Remove empty plots
for j in range(len(features_plot), len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    'Feature Distribution by Cluster',
    fontsize=14,
    fontweight='bold'
)

plt.tight_layout()
plt.savefig('cluster_distribution.png', bbox_inches='tight')
plt.show()

### 4.6 Churn Rate by Cluster

In [ ]:
churn_by_cluster = df.groupby('cluster')['Churn'].agg(['mean', 'sum', 'count'])
churn_by_cluster.columns = ['Churn Rate', 'Churned Count', 'Total Customers']
churn_by_cluster = churn_by_cluster.sort_values('Churn Rate', ascending=False)
churn_by_cluster['Churn Rate (%)'] = (churn_by_cluster['Churn Rate'] * 100).round(2)
print(churn_by_cluster)

plt.figure(figsize=(9, 5))
colors = ['#e74c3c' if x > 0.15 else '#f39c12' if x > 0.07 else '#2ecc71'
          for x in churn_by_cluster['Churn Rate']]
bars = plt.bar(churn_by_cluster.index.astype(str), churn_by_cluster['Churn Rate (%)'],
               color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, churn_by_cluster['Churn Rate (%)']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
plt.title('Churn Rate by Cluster', fontsize=13, fontweight='bold')
plt.xlabel('Cluster')
plt.ylabel('Churn Rate (%)')
plt.tight_layout()
plt.savefig('churn_by_cluster.png', bbox_inches='tight')
plt.show()

### 4.7 Cluster Profiles

In [ ]:
print("=== CLUSTER PROFILES ===\n")
for c in sorted(df['cluster'].unique()):
    sub = df[df['cluster']==c]
    churn_rate = sub['Churn'].mean()
    n = len(sub)
    print(f"--- Cluster {c} | {n} customers | Churn: {churn_rate:.1%} ---")
    print(f"  Average age:            {sub['Age'].mean():.1f} years")
    print(f"  Time as customer:       {sub['Lifetime'].mean():.1f} months")
    print(f"  Contract period:        {sub['Contract_period'].mean():.1f} months")
    print(f"  Months to expiration:   {sub['Month_to_end_contract'].mean():.1f}")
    print(f"  Visit freq. (total):    {sub['Avg_class_frequency_total'].mean():.2f}/week")
    print(f"  Visit freq. (current):  {sub['Avg_class_frequency_current_month'].mean():.2f}/week")
    print(f"  Additional charges:     ${sub['Avg_additional_charges_total'].mean():.0f}")
    print(f"  Group visits:           {sub['Group_visits'].mean():.0%}")
    print(f"  Near location:          {sub['Near_Location'].mean():.0%}")
    print(f"  Partner company:        {sub['Partner'].mean():.0%}")
    print()

## Step 5. Conclusions and Recommendations

### 5.1 Main Churn Drivers

Based on the exploratory analysis, predictive models, and clustering, the factors that most influence customer **churn** are:

| Factor | Effect |
|---|---|
| **Short contract period (1 month)** | High risk — customers without a long-term commitment leave easily |
| **Low visit frequency** | Customers who attend less often tend to cancel |
| **Short relationship tenure (low Lifetime)** | New customers are the most vulnerable |
| **Contract close to expiration** | Risk increases as the contract nears its end date |
| **Not participating in group sessions** | Group class members have a stronger social bond with the gym |
| **Not from a partner company / no promo_friends** | Customers without an institutional tie churn more |

---

### 5.2 Strategic Recommendations

**1. Encourage long-term contracts from the start**
- Offer a progressive discount (e.g., 10% for 3 months, 20% for 12 months) at sign-up.
- Implement a loyalty program with a bonus after 6 months of membership.

**2. Proactive alerts for low-frequency customers**
- Customers whose visits drop below 1x/week should receive active outreach (email, push notification, phone call).
- Offer a free session with a personal trainer as a "reactivation" before the customer churns.

**3. Social engagement — expand group classes**
- Customers who take part in group activities have a significantly lower churn rate.
- Create monthly group challenges, let members connect with friends on the platform, and promote themed classes.

**4. Special intervention for first-quarter customers (Lifetime 0–3 months)**
- The first 3 months are critical. Create a "welcome program" with monthly check-ins and personalized goals.
- Assign a dedicated instructor for the first 30 days to build rapport and habit.

**5. Early renewal alerts**
- Notify customers 30 days before contract expiration with personalized renewal offers.
- For high-risk customers (as predicted by the model), offer an extra incentive (a free month, a discount) to secure renewal.

# Final Conclusion — Churn Analysis (Fitness)

The analysis was carried out on a total of {len(df):,} customers, with an overall churn rate of {df['Churn'].mean():.2%}, indicating the percentage of customers who cancelled or left the service during the analyzed period.

Two predictive models were developed to estimate churn:

### Logistic Regression Model

The model produced the following results:

Accuracy: {acc_lr:.4f}
Precision: {prec_lr:.4f}
Recall: {rec_lr:.4f}

This model offers good interpretability and consistent performance, making it useful for understanding the main factors associated with customer churn.

### Random Forest Model

The Random Forest model produced:

Accuracy: {acc_rf:.4f}
Precision: {prec_rf:.4f}
Recall: {rec_rf:.4f}

This model tends to capture more complex relationships in the data, generally showing greater robustness and better predictive power.

### Cluster Analysis

Customer segmentation revealed important differences in churn rate across groups. Some clusters show a significantly higher cancellation risk than others, highlighting distinct customer behavior profiles.

Clusters were ranked by churn rate in descending order, making it possible to identify which groups should be prioritized in retention strategies.

# Overall Conclusion

Overall, the results indicate a clear risk-segmentation pattern among customers. The models developed are able to predict churn with satisfactory performance, and the cluster analysis provides important strategic insights for more targeted retention actions.

Combining predictive modeling with segmentation allows the company to act more efficiently, focusing efforts on the groups most likely to cancel.